In [ ]:
import sys, os, io, time, random, contextlib, urllib.request
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader
from seqeval.metrics import classification_report
from seqeval.metrics.sequence_labeling import get_entities

sys.path.append("..")
from src.data import read_conll, get_paths
from src.evaluate import evaluate

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

# GPU si hay (Colab); si no, CPU. Este modelo entrena bien en CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

os.makedirs("../data", exist_ok=True)
os.makedirs("../models", exist_ok=True)
os.makedirs("../results", exist_ok=True)

In [ ]:
paths = get_paths()
train = read_conll(paths["train"])
dev = read_conll(paths["dev"])
test = read_conll(paths["test"])
print(f"train: {len(train)}  dev: {len(dev)}  test: {len(test)} oraciones")

In [ ]:
PAD, UNK = "<PAD>", "<UNK>"
PAD_IDX, UNK_IDX = 0, 1

word_counts = Counter(w.lower() for tokens, _ in train for w in tokens)
word2idx = {PAD: PAD_IDX, UNK: UNK_IDX}
for w in word_counts:
    word2idx[w] = len(word2idx)

all_tags = sorted({t for split in (train, dev, test) for _, tags in split for t in tags})
tag2idx = {t: i for i, t in enumerate(all_tags)}
idx2tag = {i: t for t, i in tag2idx.items()}

def encode(tokens):
    return [word2idx.get(w.lower(), UNK_IDX) for w in tokens]

print(f"Vocabulario: {len(word2idx):,} palabras")
print("Etiquetas:", all_tags)

In [ ]:
from gensim.models import KeyedVectors

FT_PATH = "../data/cc.es.300.vec.gz"
FT_URL = "https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.es.300.vec.gz"

if not os.path.exists(FT_PATH):
    print("Descargando FastText, puede tardar varios minutos...")
    urllib.request.urlretrieve(FT_URL, FT_PATH)

t0 = time.perf_counter()
kv = KeyedVectors.load_word2vec_format(FT_PATH, limit=300_000)
print(f"FastText cargado en {time.perf_counter() - t0:.0f} s, dimensión {kv.vector_size}")

In [ ]:
EMB_DIM = kv.vector_size
rng = np.random.default_rng(SEED)

def random_matrix():
    m = rng.uniform(-0.25, 0.25, (len(word2idx), EMB_DIM)).astype(np.float32)
    m[PAD_IDX] = 0.0
    return m

# Matriz FastText: vector preentrenado si la palabra existe, aleatorio si no
ft_matrix = random_matrix()
found = 0
for w, i in word2idx.items():
    if i > UNK_IDX and w in kv.key_to_index:
        ft_matrix[i] = kv[w]
        found += 1

rand_matrix = random_matrix()

n_words = len(word2idx) - 2
print(f"Cobertura de FastText sobre el vocabulario: {found:,}/{n_words:,} ({found / n_words:.1%})")

In [ ]:
total, unk = Counter(), Counter()
for tokens, tags in test:
    for etype, start, end in get_entities(tags):
        if etype == "UNCLEAR":
            continue
        total[etype] += 1
        if any(word2idx.get(w.lower(), UNK_IDX) == UNK_IDX for w in tokens[start:end + 1]):
            unk[etype] += 1

unk_table = pd.DataFrame({"entidades": total, "con_UNK": unk}).fillna(0).astype(int)
unk_table.loc["TOTAL"] = unk_table.sum()
unk_table["%_UNK"] = (100 * unk_table["con_UNK"] / unk_table["entidades"]).round(1)
unk_table

In [ ]:
def make_loader(sentences, batch_size=32, shuffle=False):
    data = [(encode(tokens), [tag2idx[t] for t in tags]) for tokens, tags in sentences]

    def collate(batch):
        lengths = torch.tensor([len(x) for x, _ in batch])
        max_len = int(lengths.max())
        X = torch.full((len(batch), max_len), PAD_IDX, dtype=torch.long)
        Y = torch.full((len(batch), max_len), -100, dtype=torch.long)
        for i, (x, y) in enumerate(batch):
            X[i, :len(x)] = torch.tensor(x)
            Y[i, :len(y)] = torch.tensor(y)
        return X, Y, lengths

    return DataLoader(data, batch_size=batch_size, shuffle=shuffle, collate_fn=collate)

X, Y, lengths = next(iter(make_loader(train)))
print("X:", tuple(X.shape), " Y:", tuple(Y.shape), " largos:", lengths[:8].tolist())

In [ ]:
class BiLSTMTagger(nn.Module):
    def __init__(self, emb_matrix, n_tags, hidden=256, dropout=0.3, freeze=False):
        super().__init__()
        self.emb = nn.Embedding.from_pretrained(
            torch.tensor(emb_matrix), freeze=freeze, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(emb_matrix.shape[1], hidden, num_layers=2,
                            bidirectional=True, batch_first=True, dropout=dropout)
        self.drop = nn.Dropout(dropout)
        self.out = nn.Linear(2 * hidden, n_tags)

    def forward(self, x, lengths):                      # x: (batch, seq_len)
        e = self.drop(self.emb(x))                      # (batch, seq_len, 300)
        packed = pack_padded_sequence(e, lengths.cpu(), batch_first=True,
                                      enforce_sorted=False)
        h, _ = self.lstm(packed)
        h, _ = pad_packed_sequence(h, batch_first=True, total_length=x.size(1))
        return self.out(self.drop(h))                   # (batch, seq_len, n_tags)

m = BiLSTMTagger(ft_matrix, len(tag2idx))
print(f"Parámetros entrenables: {sum(p.numel() for p in m.parameters() if p.requires_grad):,}")
del m

In [ ]:
@torch.no_grad()
def predict(model, sentences):
    model.eval()
    preds = []
    for X, _, lengths in make_loader(sentences, batch_size=64):
        best = model(X.to(device), lengths).argmax(-1).cpu()
        for row, n in zip(best, lengths):
            preds.append([idx2tag[int(i)] for i in row[:n]])
    return preds

def dev_f1(model):
    """F1 en dev con src/evaluate.py, sin imprimir el reporte en cada época."""
    y_true = [tags for _, tags in dev]
    y_pred = predict(model, dev)
    with contextlib.redirect_stdout(io.StringIO()):
        return evaluate(y_true, y_pred)

In [ ]:
def train_model(name, emb_matrix, freeze=False, max_epochs=30, patience=3, lr=1e-3):
    set_seed()
    model = BiLSTMTagger(emb_matrix, len(tag2idx), freeze=freeze).to(device)
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
    train_loader = make_loader(train, shuffle=True)
    ckpt = f"../models/bilstm_{name}.pt"

    best, bad, history = -1.0, 0, []
    t0 = time.perf_counter()
    for epoch in range(1, max_epochs + 1):
        model.train()
        total_loss = 0.0
        for X, Y, lengths in train_loader:
            X, Y = X.to(device), Y.to(device)
            opt.zero_grad()
            logits = model(X, lengths)
            loss = loss_fn(logits.reshape(-1, logits.size(-1)), Y.reshape(-1))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            total_loss += loss.item()

        f1 = dev_f1(model)
        history.append({"epoch": epoch, "loss": total_loss / len(train_loader), "dev_f1": f1})
        print(f"[{name}] época {epoch:2d}  loss {history[-1]['loss']:.4f}  dev F1 {f1:.4f}")

        if f1 > best:
            best, bad = f1, 0
            torch.save(model.state_dict(), ckpt)
        else:
            bad += 1
            if bad >= patience:
                print(f"[{name}] early stopping: {patience} épocas sin mejorar")
                break

    train_time = time.perf_counter() - t0
    model.load_state_dict(torch.load(ckpt, map_location=device))
    return {"model": model, "history": pd.DataFrame(history),
            "best_dev_f1": best, "train_time": train_time}

In [ ]:
VARIANTS = {
    "random":          dict(emb_matrix=rand_matrix, freeze=False),
    "fasttext":        dict(emb_matrix=ft_matrix,   freeze=False),
    "fasttext_frozen": dict(emb_matrix=ft_matrix,   freeze=True),
}

runs = {}
for name, cfg in VARIANTS.items():
    print(f"\n=== {name} ===")
    runs[name] = train_model(name, **cfg)

In [ ]:
def drop_unclear(seqs):
    # misma regla que src/evaluate.py: UNCLEAR pasa a "O"
    return [["O" if t.endswith("UNCLEAR") else t for t in s] for s in seqs]

y_test = [tags for _, tags in test]
rows, per_type = [], {}

for name, run in runs.items():
    print(f"\n=== BiLSTM {name}: test ===")
    y_pred = predict(run["model"], test)
    evaluate(y_test, y_pred)
    rep = classification_report(drop_unclear(y_test), drop_unclear(y_pred), output_dict=True)
    rows.append({
        "modelo": f"BiLSTM ({name})",
        "precision": rep["micro avg"]["precision"],
        "recall": rep["micro avg"]["recall"],
        "f1": rep["micro avg"]["f1-score"],
        "dev_f1": run["best_dev_f1"],
        "epocas": len(run["history"]),
        "tiempo_min": run["train_time"] / 60,
    })
    per_type[f"BiLSTM ({name})"] = {k: v["f1-score"] for k, v in rep.items() if "avg" not in k}

results = pd.DataFrame(rows).round(4)
f1_by_type = pd.DataFrame(per_type).round(4)
results.to_csv("../results/bilstm_results.csv", index=False)
f1_by_type.to_csv("../results/bilstm_f1_por_tipo.csv")
results

In [ ]:
f1_by_type

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for name, run in runs.items():
    h = run["history"]
    ax.plot(h["epoch"], h["dev_f1"], marker="o", label=name)
ax.set_xlabel("Época")
ax.set_ylabel("F1 en dev (sin UNCLEAR)")
ax.set_title("BiLSTM: F1 en dev por época")
ax.grid(alpha=0.3)
ax.legend()
fig.savefig("../results/bilstm_dev_f1.png", dpi=150, bbox_inches="tight")
plt.show()